# Phase 3 - Model: cancellation-risk prediction

The brief allows **one** model: cancellation-risk **or** RFM segmentation.

## Why cancellation-risk, not RFM

RFM needs a usable *Frequency* dimension. In this data 3,489 of 4,366 diving
customers - **79.9%** - have exactly one completed booking, and
`repeat_customer_rate` is 16.0%. Frequency would be a near-constant 1 and
Recency would mostly restate "when did you visit", so RFM would produce
segments driven by a single spend variable. That is a weak basis for a
recommendation.

Cancellation, by contrast, is the operator's largest controllable leak:
`cancellation_rate` has risen 9.7% -> 18.5% -> 25.7% across the seasons and
59.6% of cancellations are *not* attributed to weather. A model that ranks
bookings by cancellation risk supports a concrete operational policy
(confirmation calls, deposits, targeted overbooking), which is exactly what
Phase 5 needs.

## Validation design

Split is **out-of-time**, not random: train on 2023-24 + 2024-25, test on
2025-26. A random split would leak same-trip and same-weather-day information
between train and test and would flatter the score. Out-of-time answers the
question the owner actually has: *does this predict the season we are in?*

In [1]:
import os
import sqlite3
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss, confusion_matrix

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
RANDOM_STATE = 42

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
CLEAN = os.path.join(ROOT, "data", "clean")
con = sqlite3.connect(os.path.join(CLEAN, "dive_ops_clean.db"))
bk = pd.read_sql("SELECT * FROM bookings", con, parse_dates=["booking_date", "dive_date"])
cu = pd.read_sql("SELECT * FROM customers", con)
tp = pd.read_sql("SELECT * FROM trips", con)
wx = pd.read_sql("SELECT * FROM weather", con, parse_dates=["weather_date"])
con.close()

d = (bk.merge(tp, on="trip_id", how="left")
       .merge(cu[["customer_id", "cert_level", "acquisition_channel", "age"]],
              on="customer_id", how="left")
       .merge(wx.rename(columns={"weather_date": "dive_date"}), on="dive_date", how="left"))

## Target and leakage control

Target = **booking was cancelled** (Cancelled vs Completed + No-show). A
no-show is a different failure with a different remedy, so it stays in the
negative class rather than being merged into the positive one.

`cancellation_reason` is dropped outright: it only exists *because* the booking
cancelled. `status` is the target. `price_quarantined` and `revenue_inr` are
also dropped - `revenue_inr` is null exactly when the booking did not complete,
which is near-perfect leakage.

In [2]:
d["y"] = (d["status"] == "Cancelled").astype(int)
print("target base rate: %.1f%% (%d of %d)" % (100 * d["y"].mean(), d["y"].sum(), len(d)))

LEAKY = ["status", "cancellation_reason", "revenue_inr", "price_quarantined",
         "booking_id", "customer_id", "trip_id", "boat_slot", "dive_ym"]
CAT = ["sea_condition", "dive_type", "acquisition_channel", "season", "difficulty",
       "cert_level"]
NUM = ["lead_time_days", "num_dives", "price_inr", "boat_capacity", "dive_month",
       "is_weekend", "age"]

d["is_weekend"] = d["is_weekend"].astype(int)
print("\nfeatures: %d categorical, %d numeric" % (len(CAT), len(NUM)))
print("nulls in numeric features:\n", d[NUM].isna().sum().to_string())

train = d[d["dive_date"] <= "2025-05-31"].copy()
test = d[d["dive_date"] > "2025-05-31"].copy()
print("\ntrain: %d bookings (%s -> %s), cancel rate %.1f%%"
      % (len(train), train.dive_date.min().date(), train.dive_date.max().date(),
         100 * train.y.mean()))
print("test : %d bookings (%s -> %s), cancel rate %.1f%%"
      % (len(test), test.dive_date.min().date(), test.dive_date.max().date(),
         100 * test.y.mean()))
print("\nNote the base rate shifts between train and test (%.1f%% -> %.1f%%); this is"
      " the deterioration the model has to cope with." % (100 * train.y.mean(),
                                                          100 * test.y.mean()))

target base rate: 18.6% (1333 of 7171)

features: 6 categorical, 7 numeric
nulls in numeric features:
 lead_time_days      0
num_dives           0
price_inr          23
boat_capacity       0
dive_month          0
is_weekend          0
age               565

train: 4775 bookings (2024-01-01 -> 2025-05-31), cancel rate 15.0%
test : 2396 bookings (2025-06-03 -> 2026-05-31), cancel rate 25.7%

Note the base rate shifts between train and test (15.0% -> 25.7%); this is the deterioration the model has to cope with.


## Fit

Logistic regression, chosen over a tree ensemble deliberately: the deliverable
is a *policy recommendation*, so signed, readable odds ratios are worth more
than a couple of points of AUC. Median-imputed numerics, one-hot categoricals.

In [3]:
pre = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first",
                          sparse_output=False), CAT),
])
model = Pipeline([("pre", pre),
                  ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))])
model.fit(train[CAT + NUM], train["y"])

p_test = model.predict_proba(test[CAT + NUM])[:, 1]
p_train = model.predict_proba(train[CAT + NUM])[:, 1]

auc_test = roc_auc_score(test["y"], p_test)
auc_train = roc_auc_score(train["y"], p_train)
print("ROC AUC  train %.3f | test (out-of-time) %.3f" % (auc_train, auc_test))
print("Brier    test %.4f  (base-rate-only baseline %.4f)"
      % (brier_score_loss(test["y"], p_test),
         brier_score_loss(test["y"], np.repeat(train["y"].mean(), len(test)))))

ROC AUC  train 0.739 | test (out-of-time) 0.749
Brier    test 0.1578  (base-rate-only baseline 0.2024)


## Does weather alone explain it?

The central question for the memo. Fit a weather-only model and compare. If
weather were the whole story the two AUCs would be close.

In [4]:
wx_only = Pipeline([
    ("pre", ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore",
                                                     drop="first", sparse_output=False),
                                ["sea_condition"])])),
    ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))])
wx_only.fit(train[["sea_condition"]], train["y"])
auc_wx = roc_auc_score(test["y"], wx_only.predict_proba(test[["sea_condition"]])[:, 1])

print("weather-only AUC : %.3f" % auc_wx)
print("full model AUC   : %.3f" % auc_test)
print("lift from non-weather features: +%.3f AUC (%.0f%% of the gap from 0.5 to the"
      " full model)" % (auc_test - auc_wx, 100 * (auc_test - auc_wx) / (auc_test - 0.5)))

weather-only AUC : 0.725


full model AUC   : 0.749
lift from non-weather features: +0.023 AUC (9% of the gap from 0.5 to the full model)


## What drives risk

Coefficients as odds ratios. >1 raises cancellation odds, <1 lowers them.

In [5]:
names = (NUM + list(model.named_steps["pre"].named_transformers_["cat"]
                    .get_feature_names_out(CAT)))
coef = pd.DataFrame({"feature": names,
                     "coef": model.named_steps["clf"].coef_[0]})
coef["odds_ratio"] = np.exp(coef["coef"])
coef = coef.reindex(coef["coef"].abs().sort_values(ascending=False).index)
print("all features by |coefficient| (numerics are per 1 SD):")
print(coef.round(3).to_string(index=False))

all features by |coefficient| (numerics are per 1 SD):
                      feature   coef  odds_ratio
         sea_condition_Closed  3.208      24.735
          sea_condition_Rough  1.686       5.398
  acquisition_channel_Walk-in -0.668       0.512
 acquisition_channel_Referral -0.653       0.520
       sea_condition_Moderate  0.509       1.664
      acquisition_channel_OTA  0.457       1.579
                    price_inr  0.334       1.397
  acquisition_channel_Unknown -0.326       0.722
                    num_dives -0.296       0.744
  acquisition_channel_Website -0.243       0.785
                   dive_month  0.178       1.195
     dive_type_Discovery Dive -0.166       0.847
acquisition_channel_Instagram -0.166       0.847
        cert_level_Divemaster  0.146       1.158
        cert_level_Open Water -0.122       0.885
              season_Shoulder -0.114       0.892
                   is_weekend  0.099       1.105
                  season_Peak -0.095       0.910
      difficul

## Decile lift - the operationally useful output

Rank the current season's bookings by predicted risk and check how much of the
cancellation book concentrates in the riskiest deciles. This is what a
confirmation-call or deposit policy would be targeted at.

In [6]:
t = test[["booking_id", "dive_date", "dive_type", "acquisition_channel",
          "sea_condition", "lead_time_days", "price_inr", "y"]].copy()
t["risk"] = p_test
t["decile"] = pd.qcut(t["risk"].rank(method="first"), 10,
                      labels=range(10, 0, -1)).astype(int)

lift = t.groupby("decile").agg(bookings=("y", "size"), cancellations=("y", "sum"),
                               actual_rate=("y", "mean"), mean_pred=("risk", "mean")
                               ).sort_index()
lift["cum_cancellations"] = lift["cancellations"].cumsum()
lift["cum_capture"] = lift["cum_cancellations"] / lift["cancellations"].sum()
lift["lift_vs_base"] = lift["actual_rate"] / test["y"].mean()
print("decile 1 = highest predicted risk\n")
print(lift.assign(actual_rate=(100 * lift.actual_rate).round(1),
                  mean_pred=(100 * lift.mean_pred).round(1),
                  cum_capture=(100 * lift.cum_capture).round(1),
                  lift_vs_base=lift.lift_vs_base.round(2)).to_string())

top3 = lift.loc[1:3]
print("\nTop 3 deciles = %d bookings (%.0f%% of the season) contain %d of %d "
      "cancellations (%.0f%%), at %.1fx the base rate."
      % (top3.bookings.sum(), 100 * top3.bookings.sum() / len(t),
         top3.cancellations.sum(), int(test.y.sum()),
         100 * top3.cancellations.sum() / test.y.sum(),
         top3.actual_rate.mean() / test.y.mean()))

decile 1 = highest predicted risk



        bookings  cancellations  actual_rate  mean_pred  cum_cancellations  cum_capture  lift_vs_base
decile                                                                                               
1            240            198         82.5       57.2                198         32.1          3.21
2            239             92         38.5       28.4                290         47.1          1.50
3            240             68         28.3       20.6                358         58.1          1.10
4            239             47         19.7       16.4                405         65.7          0.76
5            240             56         23.3       13.7                461         74.8          0.91
6            239             38         15.9       11.8                499         81.0          0.62
7            240             37         15.4       10.1                536         87.0          0.60
8            239             27         11.3        8.4                563        

## Confusion matrix at an operating threshold

Threshold chosen at the top 30% of risk, matching the decile analysis above -
i.e. "which bookings get a confirmation call".

In [7]:
thr = np.quantile(p_test, 0.70)
pred = (p_test >= thr).astype(int)
tn, fp, fn, tp = confusion_matrix(test["y"], pred).ravel()
print("threshold p >= %.3f (top 30%% of risk)" % thr)
print("  flagged bookings      : %d" % (tp + fp))
print("  cancellations caught  : %d of %d  (recall %.1f%%)"
      % (tp, tp + fn, 100 * tp / (tp + fn)))
print("  precision             : %.1f%%" % (100 * tp / (tp + fp)))
print("  missed cancellations  : %d" % fn)

threshold p >= 0.181 (top 30% of risk)
  flagged bookings      : 719
  cancellations caught  : 358 of 616  (recall 58.1%)
  precision             : 49.8%
  missed cancellations  : 258


## Export scored bookings

Written to `data/clean/` for the Phase 4 dashboard export so the dashboard can
carry a risk view alongside the historical KPIs.

In [8]:
t.drop(columns="y").to_csv(os.path.join(CLEAN, "cancellation_risk_scores.csv"),
                           index=False)
summary = pd.DataFrame([
    {"metric": "model", "value": "LogisticRegression (out-of-time split)"},
    {"metric": "train_rows", "value": len(train)},
    {"metric": "test_rows", "value": len(test)},
    {"metric": "train_cancel_rate", "value": round(float(train.y.mean()), 4)},
    {"metric": "test_cancel_rate", "value": round(float(test.y.mean()), 4)},
    {"metric": "auc_train", "value": round(float(auc_train), 3)},
    {"metric": "auc_test", "value": round(float(auc_test), 3)},
    {"metric": "auc_weather_only", "value": round(float(auc_wx), 3)},
    {"metric": "top3_decile_capture", "value": round(float(top3.cancellations.sum()
                                                           / test.y.sum()), 3)},
    {"metric": "recall_at_top30pct", "value": round(float(tp / (tp + fn)), 3)},
    {"metric": "precision_at_top30pct", "value": round(float(tp / (tp + fp)), 3)},
])
summary.to_csv(os.path.join(CLEAN, "model_summary.csv"), index=False)
print(summary.to_string(index=False))

               metric                                  value
                model LogisticRegression (out-of-time split)
           train_rows                                   4775
            test_rows                                   2396
    train_cancel_rate                                 0.1502
     test_cancel_rate                                 0.2571
            auc_train                                  0.739
             auc_test                                  0.749
     auc_weather_only                                  0.725
  top3_decile_capture                                  0.581
   recall_at_top30pct                                  0.581
precision_at_top30pct                                  0.498
